# 寻找预测指标的threshold
### junjiechen
### 20260513

针对预测结构，采用DockQ的方式获得不同的指标
- DockQ: 将大于0.23的结果作为成功预测的结果
- iRMSD: 小于2A
- LRMSD: 小于2.5A

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 绘制docking_results.csv中的指标散点图，横坐标为pTM，纵坐标为ipTM，点颜色为DockQ结果，绿色为DockQ>=0.23, 红色为DockQ<0.23
df = pd.read_csv('./DockQ/pro_msa_template-pep_nomsa_notemplate/docking_results.csv')
df['DockQ_category'] = df['lrmsd'].apply(lambda x: '>=2.5' if x >= 2.5 else '<2.5')
color_mapping = {'<2.5': 'green', '>=2.5': 'red'}
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='pep_plddt', y='iptm', hue='DockQ_category', palette=color_mapping)
plt.title('Scatter Plot of pep_plddt vs iptm Colored by lrmsd Category')
plt.xlabel('pep_plddt')
plt.ylabel('iptm')
plt.legend(title='lrmsd Category')
plt.grid()
plt.show()

In [ ]:
method = ['pro_msa_notemplate-pep_nomsa_notemplate', 
         'pro_msa_template-pep_nomsa_notemplate', 
         'pro_nomsa_notemplate-pep_nomsa_notemplate', 
         'pro_nomsa_template-pep_nomsa_notemplate']

metrics = ['pep_plDDT',
           'plDDT', 
           'ipTM', 
           'pTM',
           'pep_pTM',
           'ranking_score', 
           'pAE_min',
           'ipAE',
           'ipSAE_max',
           'ipSAE_min',
           'oracle'
           ]

# DockQ

In [ ]:
# 统计不同方法下，按照不同指标选择每个target的top1模型的DockQ结果，绘制柱状图，并计算均值和标准差
rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['dockq_score']
    for metric in metrics:
        if 'pAE' in metric.strip():
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        for _, row in top1.iterrows():
            rows.append({
                'method': m,
                'native_top1': row['native'],
                'dockq_score': row['dockq_score'],
                'metric': metric,
                'value': row[metric]
            })

top1_df = pd.DataFrame(rows)

plt.figure(figsize=(12, 8))
sns.barplot(data=top1_df, x='metric', y='dockq_score', hue='method', errorbar='sd')
plt.title('DockQ Scores of Top1 Models by Different Metrics and Methods')
plt.xlabel('Metric')
plt.ylabel('DockQ Score')
plt.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

In [ ]:
# 按照DockQ>=0.23作为成功预测的结果，绘制不同metric选择的top1模型的成功率柱状图
success_rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['dockq_score']
    for metric in metrics:
        if 'pAE' in metric.strip():
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        success_rate = (top1['dockq_score'] >= 0.23).mean()
        success_rows.append({
            'method': m,
            'metric': metric,
            'success_rate': success_rate
        })
success_df = pd.DataFrame(success_rows)
plt.figure(figsize=(12, 8))
sns.barplot(data=success_df, x='metric', y='success_rate', hue='method')
plt.title('Success Rate of Top1 Models (DockQ >= 0.23)')
plt.xlabel('Metric')
plt.ylabel('Success Rate')
plt.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

In [ ]:
# 只绘制DockQ >= 0.23的柱状图，绘制DockQ的均值和标准差
rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['dockq_score']
    for metric in metrics:
        if 'pAE' in metric.strip():
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        filtered = top1[top1['dockq_score'] >= 0.23]
        for _, row in filtered.iterrows():
            rows.append({
                'method': m,
                'metric': metric,
                'ldockq_score': row['dockq_score']
            })

top1_df = pd.DataFrame(rows)
plt.figure(figsize=(12, 8))
sns.barplot(data=top1_df, x='metric', y='ldockq_score', hue='method', errorbar='sd')
plt.title('Mean DockQ of Top1 Models (DockQ >= 0.23)')
plt.xlabel('Metric')
plt.ylabel('DockQ')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

# lrmsd

In [ ]:
# 统计不同方法下，按照不同指标选择每个target的top1模型的LRMSD结果，绘制柱状图，并计算均值和标准差
rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['lRMSD']
    for metric in metrics:
        if 'pAE' in metric.strip() or metric == 'oracle':
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        for _, row in top1.iterrows():
            rows.append({
                'method': m,
                'native_top1': row['native'],
                'lRMSD': row['lRMSD'],
                'metric': metric
            })

top1_df = pd.DataFrame(rows)

plt.figure(figsize=(12, 8))
sns.barplot(data=top1_df, x='metric', y='lRMSD', hue='method', errorbar='sd')
plt.title('LRMSD of Top1 Models by Different Metrics and Methods')
plt.xlabel('Metric')
plt.ylabel('LRMSD')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

In [ ]:
# 只绘制lrmsd < 2.5的柱状图，绘制lrmsd的均值和标准差
rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['lRMSD']
    for metric in metrics:
        if 'pAE' in metric.strip() or metric == 'oracle':
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        filtered = top1[top1['lRMSD'] < 2.5]
        for _, row in filtered.iterrows():
            rows.append({
                'method': m,
                'metric': metric,
                'lRMSD': row['lRMSD']
            })

top1_df = pd.DataFrame(rows)
plt.figure(figsize=(12, 8))
sns.barplot(data=top1_df, x='metric', y='lRMSD', hue='method', errorbar='sd')
plt.title('Mean lRMSD of Top1 Models (lRMSD < 2.5)')
plt.xlabel('Metric')
plt.ylabel('lRMSD')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

In [ ]:
# 以lRMSD < 2.5作为成功预测的结果，绘制每种指标选择的top1成功率柱状图
success_rows = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['lRMSD']
    for metric in metrics:
        if 'pAE' in metric.strip() or metric == 'oracle':
            top1 = df.loc[df.groupby('native')[metric].idxmin()]
        else:
            top1 = df.loc[df.groupby('native')[metric].idxmax()]
        
        success_rate = (top1['lRMSD'] < 2.5).mean()
        success_rows.append({
            'method': m,
            'metric': metric,
            'success_rate': success_rate
        })
success_df = pd.DataFrame(success_rows)
plt.figure(figsize=(12, 8))
sns.barplot(data=success_df, x='metric', y='success_rate', hue='method')
plt.title('Success Rate of Top1 Models (lRMSD < 2.5)')
plt.xlabel('Metric')
plt.ylabel('Success Rate')
plt.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')
plt.grid()
plt.show()

In [ ]:
# 以lRMSD < 2.5作为成功预测的结果，分析每一组指标的top1以及top3成功率，写入csv文件
# top3成功率定义：每个target的top3预测中，至少有1个成功(lRMSD < 2.5)即算该target成功
results = []
for m in method:
    df = pd.read_csv(f'./summary/{m}.csv')
    df['oracle'] = df['lRMSD']
    for metric in metrics:
        ascending = 'pAE' in metric.strip() or metric == 'oracle'
        
        if ascending:
            top1_idx = df.groupby('native')[metric].idxmin()
            top3_mask = df.groupby('native')[metric].rank(method='first', ascending=True) <= 3
        else:
            top1_idx = df.groupby('native')[metric].idxmax()
            top3_mask = df.groupby('native')[metric].rank(method='first', ascending=False) <= 3
        
        top1 = df.loc[top1_idx]
        top3 = df.loc[top3_mask]
        
        top1_success_rate = (top1['lRMSD'] < 2.5).mean()
        # top3: 每个target的top3中至少有1个成功即算成功
        top3_success_rate = top3.groupby('native')['lRMSD'].apply(
            lambda x: (x < 2.5).any()
        ).mean()
        results.append({
            'method': m,
            'metric': metric,
            'top1_success_rate': top1_success_rate,
            'top3_success_rate': top3_success_rate,
        })

result_df = pd.DataFrame(results)
result_df.to_csv('./lrmsd_top1_top3_success_rate.csv', index=False)
print('Saved to lrmsd_top1_top3_success_rate.csv')
result_df